In [3]:
import pandas as pd 
df=pd.read_csv("/content/ethiopia.csv")
print(df.head())

In [ ]:
#add country column
df['country']='Ethiopia'
df.head()

: 

In [ ]:
df['Date'] = pd.to_datetime(df['YEAR'] * 1000 + df['DOY'], format='%Y%j')

: 

In [ ]:
df['Month'] = df['Date'].dt.month
df.head()

In [ ]:
import numpy as np
df = df.replace(-999, np.nan)
df.head()

In [ ]:
num_duplicates = df.duplicated().sum()
print(f"Number of duplicate rows found: {num_duplicates}")

if num_duplicates > 0:
    df.drop_duplicates(inplace=True)
    print("Duplicate rows have been removed.")
else:
    print("No duplicate rows found.")

print("Duplicate check was performed across all columns by default.")
df.head()

In [ ]:
numeric_df = df.select_dtypes(include=['number'])
display(numeric_df.describe())

Interpretation of Numeric Column Statistics
The df.describe() output provides key descriptive statistics for all numeric columns in the dataset:

Count: Shows the number of non-null entries for each column. This helps identify columns with missing values if the count is less than the total number of rows.
Mean: The average value of the column. For example, T2M (average temperature) has a mean of approximately 21.05 degrees Celsius.
Std (Standard Deviation): Measures the spread or dispersion of the data points around the mean. A higher standard deviation indicates greater variability.
Min: The minimum value observed in the column. For instance, the minimum average temperature (T2M) is 4.75.
25% (First Quartile): Represents the 25th percentile, meaning 25% of the data falls below this value.
50% (Median): The median or 50th percentile, which is the middle value when the data is sorted. It is less affected by outliers than the mean.
75% (Third Quartile): Represents the 75th percentile, meaning 75% of the data falls below this value.
Max: The maximum value observed in the column. For example, the maximum average temperature (T2M) is 30.13.
From these statistics, we can observe the ranges, central tendencies, and variability of climate-related measurements like temperature (T2M, T2M_MAX, T2M_MIN, T2M_RANGE), precipitation (PRECTOTCORR), humidity (RH2M), and wind speed (WS2M, WS2M_MAX).

In [ ]:
missing_values = df.isna().sum()
missing_percentage = (missing_values / len(df)) * 100

# Create a DataFrame for missing info
missing_info = pd.DataFrame({
    'Missing Values': missing_values,
    'Percentage (%)': missing_percentage
})

# Filter for columns with >5% missing values
high_missing = missing_info[missing_info['Percentage (%)'] > 5]

if not high_missing.empty:
    print("Columns with more than 5% missing values:")
    display(high_missing.sort_values(by='Percentage (%)', ascending=False))
    print("Note: High missing values in these columns could indicate data collection issues, sensor failures, or regional data gaps, potentially biasing analysis if not handled (e.g., via imputation or exclusion).")
else:
    print("No columns have more than 5% missing values.")

Outlier Detection & Basic Cleaning

In [ ]:
from scipy.stats import zscore

# Select the columns for Z-score computation
columns_to_check = ['T2M', 'T2M_MAX', 'T2M_MIN', 'PRECTOTCORR', 'RH2M', 'WS2M', 'WS2M_MAX']

z_scores = df[columns_to_check].apply(zscore)

outlier_flags = (z_scores.abs() > 3).any(axis=1)

outlier_count = outlier_flags.sum()

print(f"Number of rows with outliers (|Z| > 3) in specified columns: {outlier_count}")

## Outlier Treatment Strategy
**Decision:** Retain all 132 identified outliers.

**Reasoning:**
* **Scientific Relevance:** In climate datasets, outliers often represent genuine extreme weather events (e.g., peak temperatures or heavy precipitation). Removing them would result in a loss of critical information regarding climate volatility.
* **Data Integrity:** There is no evidence to suggest these points are the result of sensor errors or manual data entry mistakes. 
* **Modeling Impact:** Retaining these points ensures the model accounts for "tail risks" and accurately reflects the real-world variance of the Ethiopian climate.

In [ ]:

# 1. Drop rows with missing values in more than 30% of the columns
limit = int(len(df.columns) * 0.7)
df = df.dropna(thresh=limit)

# 2. Apply forward-fill to the remaining missing weather variables
df = df.ffill()

## Handling Missing Values
**Strategy:** Hybrid approach using row-wise deletion and Forward-Fill (ffill).

**1. Row Deletion (30% Threshold):**
Rows missing more than 30% of their features were removed. When a significant portion of 
a record is missing, imputation becomes unreliable and may introduce synthetic noise 
rather than capturing actual environmental trends.

**2. Forward-Fill (ffill) for Weather Variables:**
For the remaining missing values, I applied a forward-fill strategy. 
* **Reasoning:** Climate data is chronological and highly autocorrelated. Weather 
conditions at time $t$ are typically the most accurate predictors for time $t+1$. 
Forward-filling preserves this temporal continuity without shifting the mean 
as drastically as mean-imputation might.

Time series Analysis 

In [ ]:
import matplotlib.pyplot as plt


# Create a datetime index
if 'YEAR' in df.columns and 'MO' in df.columns:
    df['Date'] = pd.to_datetime(df[['YEAR', 'MO']].assign(DAY=1))
else:
    # Adjust this if your date column has a different name
    df['Date'] = pd.to_datetime(df['Date'])

df.set_index('Date', inplace=True)

# 2. Resample to Monthly Average
monthly_t2m = df['T2M'].resample('M').mean()

# 3. Identify Warmest and Coolest Months
warmest_month = monthly_t2m.idxmax()
warmest_val = monthly_t2m.max()

coolest_month = monthly_t2m.idxmin()
coolest_val = monthly_t2m.min()

# 4. Plotting
plt.figure(figsize=(14, 6))
plt.plot(monthly_t2m.index, monthly_t2m.values, color='teal', linewidth=2, label='Monthly Avg T2M')

# Annotate Warmest
plt.annotate(f'Warmest: {warmest_val:.1f}°C\n({warmest_month.strftime("%b %Y")})', 
             xy=(warmest_month, warmest_val), 
             xytext=(warmest_month, warmest_val + 1),
             arrowprops=dict(facecolor='red', shrink=0.05, width=1, headwidth=8),
             horizontalalignment='center', color='red', fontweight='bold')

# Annotate Coolest
plt.annotate(f'Coolest: {coolest_val:.1f}°C\n({coolest_month.strftime("%b %Y")})', 
             xy=(coolest_month, coolest_val), 
             xytext=(coolest_month, coolest_val - 2),
             arrowprops=dict(facecolor='blue', shrink=0.05, width=1, headwidth=8),
             horizontalalignment='center', color='blue', fontweight='bold')

plt.title('Monthly Average Temperature (T2M) in Ethiopia (2015–2026)', fontsize=14)
plt.xlabel('Year', fontsize=12)
plt.ylabel('Temperature (°C)', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.7)
plt.legend()
plt.tight_layout()

plt.savefig('monthly_t2m_plot.png')
plt.show()

In [ ]:
me=10
print(me)